# Baseline — U-Net ResNet34 Training and Diagnostic Evaluation

**Environment**: Local CPU (training) or Colab

**Part 1**: U-Net ResNet34 trained with FIRMS labels → IoU = 0.013
**Part 2**: Pixel-level evaluation — why FIRMS labels fail

## 1. Setup — Colab + Drive

In [ ]:
# Detectar si corre en Colab (incluso cuando se lanza desde VS Code)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'IN_COLAB = {IN_COLAB}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'segmentation-models-pytorch', 'albumentations'], check=True)
    print('Packages installed.')

## 2. Imports y configuración

In [ ]:
import os, random, subprocess
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss
from sklearn.model_selection import train_test_split

# ── Reproducibilidad ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('..').resolve()  # repo root

CKPT_DIR = BASE / 'models'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Copiar patches a SSD local de Colab para I/O 4x más rápido
if IN_COLAB:
    LOCAL = Path('/content/patches')
    if not (LOCAL / 'images').exists():
        print('Copiando patches a /content/ (una vez, ~10 min)...')
        LOCAL.mkdir(exist_ok=True)
        subprocess.run(['cp', '-r', str(BASE / 'data' / 'patches' / 'images'), str(LOCAL)], check=True)
        subprocess.run(['cp', '-r', str(BASE / 'data' / 'patches' / 'masks'),  str(LOCAL)], check=True)
        print('Copy complete.')
    else:
        print('Patches already in /content/ — skipping copy.')
    IMG_DIR  = LOCAL / 'images'
    MASK_DIR = LOCAL / 'masks'
else:
    IMG_DIR  = BASE / 'data' / 'patches' / 'images'
    MASK_DIR = BASE / 'data' / 'patches' / 'masks'

# ── Hiperparámetros ───────────────────────────────────────────────────────────
IN_CHANNELS  = 11
BATCH_SIZE   = 16
NUM_EPOCHS   = 40
LR           = 3e-4
VAL_FRAC     = 0.2
FIRE_WEIGHT  = 50.0   # subido de 10 → 50: ~9 fire patches por batch (antes ~3)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device   : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'IMG_DIR  : {IMG_DIR}')
print(f'CKPT_DIR : {CKPT_DIR}')
print(f'Epochs   : {NUM_EPOCHS}')
print(f'FireW    : {FIRE_WEIGHT}')

## 3. Dataset

In [ ]:
class WildfireDataset(Dataset):
    """Patches 256×256 de Sentinel-2 + máscara binaria de fuego activo."""

    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)   # (11, 256, 256)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)  # (256, 256)

        # Normalización: reflectancias y índices → [0,1] / [-1,1]
        img[:7]  /= 10000.0   # bandas espectrales (DN S2-L2A)
        img[7:10] /= 10000.0  # NDVI, NBR, NDWI
        # img[10]: valid mask, ya es 0/1

        if self.augment:
            if random.random() > 0.5:
                img  = np.flip(img,  axis=2).copy()  # flip horizontal
                mask = np.flip(mask, axis=1).copy()
            if random.random() > 0.5:
                img  = np.flip(img,  axis=1).copy()  # flip vertical
                mask = np.flip(mask, axis=0).copy()
            if random.random() > 0.5:
                k = random.randint(1, 3)
                img  = np.rot90(img,  k, axes=(1, 2)).copy()  # rotación 90/180/270
                mask = np.rot90(mask, k, axes=(0, 1)).copy()

        return (
            torch.from_numpy(img),
            torch.from_numpy(mask).unsqueeze(0)  # (1, 256, 256)
        )


print('Dataset class definida.')

## 4. Split estratificado + DataLoaders

In [ ]:
img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths), 'Mismatch imágenes/máscaras'
print(f'Patches totales: {len(img_paths)}')

# ── Detectar patches con fuego (una sola vez) ─────────────────────────────────
print('Escaneando patches con fuego...')
fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0
    for p in tqdm(mask_paths, desc='Fire scan')
], dtype=np.int32)
n_fire = fire_flags.sum()
print(f'Patches con fuego: {n_fire}/{len(fire_flags)} ({100*n_fire/len(fire_flags):.1f}%)')

# ── Split estratificado 80/20 ─────────────────────────────────────────────────
indices = np.arange(len(img_paths))
train_idx, val_idx = train_test_split(
    indices, test_size=VAL_FRAC, stratify=fire_flags, random_state=SEED
)
print(f'Train: {len(train_idx)} patches ({fire_flags[train_idx].sum()} fuego)')
print(f'Val  : {len(val_idx)}  patches ({fire_flags[val_idx].sum()} fuego)')

# ── WeightedRandomSampler ─────────────────────────────────────────────────────
train_fire = fire_flags[train_idx]
sample_weights = np.where(train_fire == 1, FIRE_WEIGHT, 1.0)
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(train_idx),
    replacement=True
)

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = WildfireDataset(
    [img_paths[i]  for i in train_idx],
    [mask_paths[i] for i in train_idx],
    augment=True
)
val_ds = WildfireDataset(
    [img_paths[i]  for i in val_idx],
    [mask_paths[i] for i in val_idx],
    augment=False
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)   # 2 workers — evita freeze en Colab
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'\nTrain batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')

## 5. Modelo — U-Net ResNet34

In [ ]:
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',  # transfer learning desde ImageNet
    in_channels=IN_CHANNELS,
    classes=1,
    activation=None              # sigmoid aplicado en la loss
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {n_params:,}')

# Verificar forward pass
with torch.no_grad():
    dummy = torch.zeros(2, IN_CHANNELS, 256, 256).to(DEVICE)
    out   = model(dummy)
    print(f'Output shape: {out.shape}')  # (2, 1, 256, 256)

## 6. Loss, optimizador y scheduler

In [ ]:
# DiceLoss + FocalLoss: combinación robusta para segmentación con clases desbalanceadas
dice_loss  = DiceLoss(mode='binary', from_logits=True)
focal_loss = FocalLoss(mode='binary', gamma=2.0, alpha=0.75)

def criterion(pred, target):
    return dice_loss(pred, target) + focal_loss(pred, target)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print('Loss: DiceLoss + FocalLoss(γ=2, α=0.75)')
print(f'Optimizer: AdamW (lr={LR})')
print(f'Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})')

## 7. Entrenamiento

In [ ]:
def fire_iou(logits, target, threshold=0.5):
    """IoU real para clase fuego.
    Ignora batches donde ni GT ni predicción tienen fuego (evita métrica inflada).
    """
    pred = (logits.sigmoid() > threshold).float()
    if target.sum() == 0 and pred.sum() == 0:
        return None  # batch sin fuego y sin FP → trivialmente correcto, no aporta info
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    return (tp / (tp + fp + fn + 1e-6)).item()


best_val_iou = 0.0
train_losses, val_losses, val_ious = [], [], []

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    ep_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch:02d}/{NUM_EPOCHS} [train]', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()

    scheduler.step()
    train_losses.append(ep_loss / len(train_loader))

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_iou_list = 0.0, []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'Epoch {epoch:02d}/{NUM_EPOCHS} [val]', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds  = model(imgs)
            v_loss += criterion(preds, masks).item()
            iou = fire_iou(preds, masks)
            if iou is not None:
                v_iou_list.append(iou)

    val_losses.append(v_loss / len(val_loader))
    val_ious.append(sum(v_iou_list) / len(v_iou_list) if v_iou_list else 0.0)

    print(f'Epoch {epoch:02d} | '
          f'Train Loss: {train_losses[-1]:.4f} | '
          f'Val Loss: {val_losses[-1]:.4f} | '
          f'Val IoU (fire): {val_ious[-1]:.4f}')

    if val_ious[-1] > best_val_iou:
        best_val_iou = val_ious[-1]
        torch.save(model.state_dict(), CKPT_DIR / 'best_unet_wildfire.pth')
        print(f'  → Modelo guardado (IoU: {best_val_iou:.4f})')

print(f'\nEntrenamiento completo. Mejor Val IoU (fire): {best_val_iou:.4f}')
print(f'Modelo en: {CKPT_DIR / "best_unet_wildfire.pth"}')

## 8. Curvas de entrenamiento

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, label='Train Loss')
axes[0].plot(epochs, val_losses,   label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (DiceLoss + FocalLoss)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, val_ious, color='green', label='Val IoU')
axes[1].axhline(best_val_iou, color='red', linestyle='--', label=f'Best IoU: {best_val_iou:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('IoU')
axes[1].set_title('Validation IoU (Fire class)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
out_fig = BASE / 'results' / 'training_curves.png'
out_fig.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_fig}')

## 9. Visualización de predicciones

In [ ]:
# Cargar mejor modelo
model.load_state_dict(torch.load(CKPT_DIR / 'best_unet_wildfire.pth', map_location=DEVICE))
model.eval()

# Seleccionar patches con fuego del set de validación
val_fire_idx = [i for i, f in enumerate(fire_flags[val_idx]) if f == 1][:4]
if not val_fire_idx:
    val_fire_idx = list(range(min(4, len(val_idx))))
    print('No hay patches con fuego en val — mostrando patches aleatorios')

fig, axes = plt.subplots(len(val_fire_idx), 4, figsize=(16, 4 * len(val_fire_idx)))
if len(val_fire_idx) == 1:
    axes = axes[np.newaxis, :]

for row, vi in enumerate(val_fire_idx):
    img_t, mask_t = val_ds[vi]
    with torch.no_grad():
        pred_logit = model(img_t.unsqueeze(0).to(DEVICE))
        pred_prob  = pred_logit.sigmoid().squeeze().cpu().numpy()
        pred_bin   = (pred_prob > 0.5).astype(np.uint8)

    img_np  = img_t.numpy()         # (11, 256, 256)
    mask_np = mask_t.squeeze().numpy()

    # RGB: B04, B03, B02 (canales 2, 1, 0)
    def norm_band(x, p=2):
        lo, hi = np.percentile(x[x > 0], [p, 100-p]) if (x > 0).any() else (0, 1)
        return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

    rgb = np.dstack([norm_band(img_np[2]), norm_band(img_np[1]), norm_band(img_np[0])])

    axes[row, 0].imshow(rgb); axes[row, 0].set_title('RGB (B04/B03/B02)'); axes[row, 0].axis('off')
    axes[row, 1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1); axes[row, 1].set_title('Ground truth'); axes[row, 1].axis('off')
    axes[row, 2].imshow(pred_prob, cmap='Reds', vmin=0, vmax=1); axes[row, 2].set_title('Predicción (prob)'); axes[row, 2].axis('off')

    overlay = rgb.copy()
    overlay[pred_bin == 1] = [1, 0.2, 0]
    axes[row, 3].imshow(overlay); axes[row, 3].set_title('RGB + predicción'); axes[row, 3].axis('off')

plt.suptitle(f'U-Net ResNet34 — Wildfire Detection | Best Val IoU: {best_val_iou:.4f}', fontsize=14)
plt.tight_layout()
out_pred = BASE / 'results' / 'predictions_sample.png'
plt.savefig(out_pred, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_pred}')

---

## Part 2 — Pixel-Level Evaluation and Diagnostic

# Phase 4 — Evaluation: Wildfire Segmentation (U-Net ResNet34)

**Environment**: local, CPU (geoai-wildfire)  
**Input**: `data/patches/` + `models/best_unet_wildfire.pth`  
**Output**: métricas completas + figuras en `results/`

| Metric | Description |
|---------|-------------|
| IoU (fire) | Intersection over Union — fire class |
| Precision | Of all predicted as fire, what % is correct |
| Recall | Of all real fire pixels, what % we detect |
| F1 | Harmonic mean of Precision/Recall |
| AP | Area under the Precision-Recall curve |

In [ ]:
import os, random, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score
)
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
IMG_DIR    = Path('d:/GeoAI/wildfire-spread/data/patches/images')
MASK_DIR   = Path('d:/GeoAI/wildfire-spread/data/patches/masks')
MODEL_PATH = Path('G:/Mon Drive/GeoAI/wildfire-spread/models/best_unet_wildfire.pth')
RESULTS    = Path('G:/Mon Drive/GeoAI/wildfire-spread/results')
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE      = torch.device('cpu')
IN_CHANNELS = 11
VAL_FRAC    = 0.2
THRESHOLD   = 0.5
BEST_IOU    = 0.5014  # registrado en Phase 3

assert MODEL_PATH.exists(), f'Modelo no encontrado: {MODEL_PATH}'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_PATH.name}  ✓')
print(f'Results: {RESULTS}')

In [ ]:
# ── Dataset — idéntico al training ────────────────────────────────────────────
class WildfireDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)
        img[:7]   /= 10000.0
        img[7:10] /= 10000.0
        return (
            torch.from_numpy(img),
            torch.from_numpy(mask).unsqueeze(0)
        )


# ── Recrear el mismo split que en training (SEED=42) ─────────────────────────
img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths), 'Mismatch images/masks'
print(f'Patches totales: {len(img_paths)}')

print('Scanning masks...')
fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0
    for p in tqdm(mask_paths, desc='Fire scan')
], dtype=np.int32)

indices = np.arange(len(img_paths))
_, val_idx = train_test_split(
    indices, test_size=VAL_FRAC, stratify=fire_flags, random_state=SEED
)

val_fire_flags = fire_flags[val_idx]
n_fire_val = val_fire_flags.sum()
print(f'Val: {len(val_idx)} patches  |  {n_fire_val} fire  |  {len(val_idx)-n_fire_val} sin fuego')

val_ds = WildfireDataset(
    [img_paths[i]  for i in val_idx],
    [mask_paths[i] for i in val_idx]
)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

In [ ]:
# ── Cargar modelo ─────────────────────────────────────────────────────────────
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=IN_CHANNELS,
    classes=1,
    activation=None
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded ✓  |  {n_params/1e6:.1f}M parameters')

In [ ]:
# ── Inferencia completa: pixel-level + per-patch ───────────────────────────────
all_probs   = []
all_targets = []
patch_ious  = []
patch_probs  = []   # guardar para visualizaciones
patch_masks  = []
patch_imgs   = []

print('Running inference (CPU, may take 5-10 min)...')
with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='Evaluating'):
        logits = model(imgs.to(DEVICE))               # (B,1,256,256)
        probs  = logits.sigmoid().squeeze(1).numpy()  # (B,256,256)
        masks_np = masks.squeeze(1).numpy()           # (B,256,256)

        all_probs.append(probs.reshape(-1))
        all_targets.append(masks_np.reshape(-1))

        # IoU por patch
        for b in range(len(imgs)):
            p = (probs[b] > THRESHOLD).astype(np.float32)
            m = masks_np[b]
            tp = (p * m).sum()
            fp = (p * (1-m)).sum()
            fn = ((1-p) * m).sum()
            patch_ious.append(float(tp / (tp + fp + fn + 1e-6)))
            patch_probs.append(probs[b])
            patch_masks.append(masks_np[b])
            patch_imgs.append(imgs[b].numpy())

all_probs   = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets).astype(np.int32)
all_preds   = (all_probs > THRESHOLD).astype(np.int32)
patch_ious  = np.array(patch_ious)

print(f'Pixels evaluados     : {len(all_targets):>12,}')
print(f'Pixels fuego (GT)    : {all_targets.sum():>12,}  ({100*all_targets.mean():.2f}%)')
print(f'Pixels pred. fuego   : {all_preds.sum():>12,}  ({100*all_preds.mean():.2f}%)')

In [ ]:
# ── Métricas ──────────────────────────────────────────────────────────────────
precision, recall, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, pos_label=1, average='binary', zero_division=0
)
cm = confusion_matrix(all_targets, all_preds)
tn, fp, fn, tp = cm.ravel()

iou_fire = tp / (tp + fp + fn + 1e-6)
iou_bg   = tn / (tn + fp + fn + 1e-6)
mean_iou = (iou_fire + iou_bg) / 2
accuracy = (tp + tn) / (tp + fp + fn + tn)
ap_score = average_precision_score(all_targets, all_probs)

# IoU sobre patches con fuego solamente
fire_patch_idx = [i for i, f in enumerate(val_fire_flags) if f == 1]
fire_patch_ious = patch_ious[fire_patch_idx]

print('=' * 52)
print('  EVALUATION METRICS — U-Net ResNet34')
print('  Dataset: Corrientes Argentina, Dic 2021–Feb 2022')
print('=' * 52)
print(f'  Precision (fire)   :  {precision:.4f}')
print(f'  Recall (fire)      :  {recall:.4f}')
print(f'  F1-Score (fire)    :  {f1:.4f}')
print(f'  IoU (fire class)   :  {iou_fire:.4f}')
print(f'  IoU (background)   :  {iou_bg:.4f}')
print(f'  Mean IoU           :  {mean_iou:.4f}')
print(f'  Avg Precision (AP) :  {ap_score:.4f}')
print(f'  Accuracy           :  {accuracy:.4f}')
print('─' * 52)
print(f'  Fire-patch IoU mean:  {fire_patch_ious.mean():.4f}')
print(f'  Fire-patch IoU std :  {fire_patch_ious.std():.4f}')
print(f'  Fire-patch IoU >0.5:  {(fire_patch_ious > 0.5).sum()}/{len(fire_patch_ious)}')
print('─' * 52)
print(f'  TP: {tp:>12,}  |  FP: {fp:>12,}')
print(f'  FN: {fn:>12,}  |  TN: {tn:>12,}')
print('=' * 52)

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

labels = ['No fire', 'Fire']
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
ax.set_xlabel('Predicción');  ax.set_ylabel('Ground Truth')
ax.set_title('Confusion Matrix (row-normalized)')

for i in range(2):
    for j in range(2):
        raw = cm[i, j]
        pct = cm_norm[i, j]
        color = 'white' if pct > 0.5 else 'black'
        ax.text(j, i, f'{pct:.2%}\n({raw:,})', ha='center', va='center',
                color=color, fontsize=9)

plt.tight_layout()
out = RESULTS / 'confusion_matrix.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Precision-Recall curve ────────────────────────────────────────────────────
prec_curve, rec_curve, _ = precision_recall_curve(all_targets, all_probs)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rec_curve, prec_curve, color='darkorange', lw=2,
        label=f'U-Net ResNet34 (AP = {ap_score:.3f})')
ax.axhline(all_targets.mean(), color='navy', linestyle='--', lw=1,
           label=f'Baseline (clase positiva = {all_targets.mean():.3f})')
ax.axvline(recall, color='green', linestyle=':', lw=1.5,
           label=f'Threshold 0.5 → Recall={recall:.3f}, Prec={precision:.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Fire Class')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])

plt.tight_layout()
out = RESULTS / 'pr_curve.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Mejores predicciones (top 3 fire patches por IoU) ─────────────────────────
def norm_band(x, p=2):
    vals = x[x > 0]
    if len(vals) == 0:
        return np.zeros_like(x)
    lo, hi = np.percentile(vals, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

best_idx  = sorted(fire_patch_idx, key=lambda i: patch_ious[i], reverse=True)[:3]
worst_idx = sorted(fire_patch_idx, key=lambda i: patch_ious[i])[:3]

def plot_panel(indices, title, filename):
    n = len(indices)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row, vi in enumerate(indices):
        img_np   = patch_imgs[vi]
        mask_np  = patch_masks[vi]
        prob_np  = patch_probs[vi]
        pred_bin = (prob_np > THRESHOLD).astype(np.uint8)
        iou_v    = patch_ious[vi]

        rgb = np.dstack([norm_band(img_np[2]), norm_band(img_np[1]), norm_band(img_np[0])])

        # Error map: TP=green, FP=orange, FN=red
        error = np.zeros((*mask_np.shape, 3))
        error[(pred_bin==1) & (mask_np==1)] = [0.0, 0.8, 0.0]  # TP verde
        error[(pred_bin==1) & (mask_np==0)] = [1.0, 0.5, 0.0]  # FP naranja
        error[(pred_bin==0) & (mask_np==1)] = [0.9, 0.1, 0.1]  # FN rojo

        overlay = rgb.copy()
        overlay[pred_bin == 1] = [1.0, 0.25, 0.0]

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f'RGB  |  IoU={iou_v:.3f}'); axes[row, 0].axis('off')

        axes[row, 1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1)
        axes[row, 1].set_title('Ground truth'); axes[row, 1].axis('off')

        axes[row, 2].imshow(prob_np, cmap='hot', vmin=0, vmax=1)
        axes[row, 2].set_title('Prediction (prob)'); axes[row, 2].axis('off')

        axes[row, 3].imshow(error)
        axes[row, 3].set_title('TP=green | FP=orange | FN=red')
        axes[row, 3].axis('off')

    tp_patch = mpatches.Patch(color=[0,0.8,0], label='TP (detected ✓)')
    fp_patch = mpatches.Patch(color=[1,0.5,0], label='FP (false alarm)')
    fn_patch = mpatches.Patch(color=[0.9,0.1,0.1], label='FN (missed fire)')
    fig.legend(handles=[tp_patch, fp_patch, fn_patch], loc='lower center',
               ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    out = RESULTS / filename
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')


plot_panel(best_idx,  'Best predictions (Top 3 IoU)',   'best_predictions.png')
plot_panel(worst_idx, 'Worst predictions (Bottom 3 IoU)', 'worst_predictions.png')

In [ ]:
# ── Distribución IoU sobre patches con fuego ──────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(fire_patch_ious, bins=20, color='firebrick', edgecolor='white', alpha=0.85)
ax.axvline(fire_patch_ious.mean(), color='black', lw=1.5, linestyle='--',
           label=f'Media: {fire_patch_ious.mean():.3f}')
ax.axvline(0.5, color='green', lw=1.5, linestyle=':',
           label='Umbral 0.5')
ax.set_xlabel('IoU'); ax.set_ylabel('Patches')
ax.set_title(f'IoU Distribution — {len(fire_patch_ious)} fire patches')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
out = RESULTS / 'iou_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Resumen para README ───────────────────────────────────────────────────────
print('\n' + '='*60)
print('  README.md BLOCK')
print('='*60)
print(f'''
## Results

| Metric | Value |
|--------|-------|
| **IoU (fire class)** | **{iou_fire:.4f}** |
| Precision | {precision:.4f} |
| Recall | {recall:.4f} |
| F1-Score | {f1:.4f} |
| Average Precision (AP) | {ap_score:.4f} |
| Mean IoU (fire + bg) | {mean_iou:.4f} |

> Model: U-Net ResNet34 | Dataset: Corrientes, Argentina (Dec 2021–Feb 2022)
> Training: 40 epochs, A100 GPU | Best Val IoU: {BEST_IOU:.4f} (epoch 16)
''')
print('='*60)

print(f'\nFiguras guardadas en: {RESULTS}')
print('  - confusion_matrix.png')
print('  - pr_curve.png')
print('  - best_predictions.png')
print('  - worst_predictions.png')
print('  - iou_distribution.png')
print('  - training_curves.png   (Phase 3)')
print('  - predictions_sample.png  (Phase 3)')